# openrouteservice avoid_areas - sahada kullanilan sistem

2023 Turkiye'de MSF bunu kullandi. **Farkimiz:** onlar kapanmayi elle haritaliyor, biz modelden cikariyoruz. Bu defter kapanmalari avoid_areas poligonu olarak verip ORS'yi kiyaslar.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/06_rota
    %pip install -q openrouteservice
print('cwd:', os.getcwd())

## 1. Graf + OD

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import common as C
C.CONFIG.update(dict(
    place='Antakya, Hatay, Turkey',   # veya 'Kahramanmaras, Turkey'
    n_od_pairs=120,                    # senaryo: Antakya 126, Maras 35
))
G = C.load_graph()
closures = C.load_closures()
od = C.sample_od_pairs(G, C.CONFIG['n_od_pairs'], C.CONFIG['seed'])
print('dugum/kenar:', G.number_of_nodes(), G.number_of_edges(), ' kapanma:', len(closures))

## 2. ORS (yerel veya API anahtari)

In [ ]:
import openrouteservice as ors, numpy as np, time
client = ors.Client(base_url='http://localhost:8080/ors')  # veya key=...
from shapely.geometry import Point, mapping
avoid = mapping(__import__('shapely.ops', fromlist=['unary_union']).unary_union(
    [Point(lon, lat).buffer(0.0003) for lat, lon in closures]))
def ors_route(s, t):
    a = [G.nodes[s]['x'], G.nodes[s]['y']]; b = [G.nodes[t]['x'], G.nodes[t]['y']]
    t0 = time.perf_counter()
    try:
        r = client.directions([a, b], profile='driving-car', options={'avoid_polygons': avoid})
        dt = time.perf_counter() - t0; seg = r['routes'][0]['summary']
        return dict(travel_time_s=seg['duration'], distance_m=seg['distance'], compute_s=dt, ok=True)
    except Exception:
        return None
rs = [ors_route(s, t) for s, t in od]; ok = [x for x in rs if x]
metrics = dict(n_pairs=len(rs), failed_route_rate=1-len(ok)/len(rs), accessibility_ratio=len(ok)/len(rs),
               mean_travel_time_s=float(np.mean([x['travel_time_s'] for x in ok])) if ok else float('nan'),
               mean_distance_m=float(np.mean([x['distance_m'] for x in ok])) if ok else float('nan'),
               mean_compute_s=float(np.mean([x['compute_s'] for x in ok])) if ok else float('nan'))
C.save_result('openrouteservice_avoid', metrics); metrics